# 負荷預測：訓練、存 DB、召回

用 **24 小時前的負荷與時間**，加上**目標時間的星期、假日**，預測全館冷凍噸（Q_COOL）。不用天氣資料。

- 訓練、回測、上線都只用監控資料（CH-1、CH-2 冷凍噸）與日期時間
- 上線時：每週一重訓；每天 07:00 與每次重啟時預測當天上班時段 08~16 時，給控制建議用
- 資料期間：2026-09-17 ~ 2026-10-13（感測資料到 10/13 11:24）

In [1]:
import sys

sys.path.append("..")

import pandas as pd
import plotly.graph_objects as go

from model import recall as model_recall
from model.train import MODEL_ID, ModelTrainer, get_chiller_hourly, save_model
from research.chart_style import COLORS, NAMES, REF_COLOR, date_axis, label, show

# 資料期間（Unix 秒）；查詢不含 END_TS，設 10/14 00:00 才含 10/13 整天
START_TS = int(pd.Timestamp("2026-09-17", tz="Asia/Taipei").timestamp())
END_TS = int(pd.Timestamp("2026-10-14", tz="Asia/Taipei").timestamp())

## 0. 完整負荷曲線

每小時一點，含夜間與週末。

In [2]:
# 直接讀主機負荷；asfreq 補上停機小時，線才會斷開
df_chiller = await get_chiller_hourly(START_TS, END_TS)
hour_index = pd.to_datetime(df_chiller["ts_hour"], unit="s", utc=True).dt.tz_convert(
    "Asia/Taipei"
)
load_hourly = df_chiller.set_index(hour_index)
load_hourly = load_hourly.asfreq("1h")

fig = go.Figure(
    go.Scatter(
        x=load_hourly.index,
        y=load_hourly["Q_COOL"],
        mode="lines",
        name=NAMES["Q_COOL"],
        line=dict(color=COLORS["Q_COOL"], width=2),
    )
)
fig.update_layout(yaxis_title=label("Q_COOL"), hovermode="x unified")
date_axis(fig)
show(fig, "全館逐時冷凍噸（CH-1 + CH-2）", "每小時平均（含停機）")

## 1. 訓練模型

- 目標：1 小時後的 Q_COOL
- 特徵（8 個）：
  - **24 小時前**：負荷、小時（sin/cos）、星期、上班時段、國定假日
  - **目標時間**：星期、國定假日
- 例：用昨天 15:00 的負荷 + 今天 15:00 是星期幾、是否假日，預測今天 15:00 的負荷
- 國定假日用 `holidays` 套件（含補假，沒有補班）
- LightGBM（100 棵樹）；沒有切測試集，指標是訓練資料內（in-sample），只算有開機的小時

In [3]:
# 主機逐時資料（上一格已讀取）
print(df_chiller.columns.tolist())
print(df_chiller.head())
print(df_chiller.shape)

['ts_hour', 'Q_COOL']
      ts_hour      Q_COOL
0  1789621200  333.054054
1  1789624800  353.580000
2  1789628400  346.750000
3  1789632000  319.483333
4  1789635600   94.883333
(623, 2)


In [4]:
trainer = ModelTrainer(
    MODEL_ID,
    START_TS,
    END_TS,
)
await trainer.main()

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000238 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 164
[LightGBM] [Info] Number of data points in the train set: 599, number of used features: 8
[LightGBM] [Info] Start training from score 76.569052


## 2. 模型存入 DB

存進 `model_registry`，並在 `model_history` 留備份（保留 365 天）；用 `python -m model.load.versions` 列出或退版。

In [5]:
await save_model(trainer)
print("模型已存入 DB")

模型已存入 DB


## 3. 召回模型重算指標

指標只算有開機的小時；圖表畫全部小時。

In [6]:
recaller = model_recall.ModelRecaller(MODEL_ID, START_TS, END_TS)
await recaller.recall_model()
prediction_frame, actual_frame = await recaller.predict()

print("召回後指標：", recaller.metric)

召回後指標： {'MAE': 5.53, 'RMSE': 9.76, 'R2': 0.99}


In [7]:
# 用訓練資料重算（in-sample），R² 不代表沒看過的日子準不準
comparison = pd.DataFrame(
    {
        "實際": actual_frame["Q_COOL"],
        "預測": prediction_frame["Q_COOL_PRED"],
    }
)
# index 是「拿來預測的那一小時」，往後移 1 小時才是被預測的小時
comparison.index = comparison.index + pd.Timedelta(hours=1)

metric = recaller.metric
metric_text = (
    f"MAE {metric['MAE']} RT、RMSE {metric['RMSE']} RT、R² = {metric['R2']:.2f}"
)

comparison_line = comparison.asfreq("1h")  # 補上停機小時，線才會斷開

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=comparison_line.index,
        y=comparison_line["實際"],
        mode="lines",
        name="實際值",
        line=dict(color=COLORS["Q_COOL"], width=2),
    )
)
fig.add_trace(
    go.Scatter(
        x=comparison_line.index,
        y=comparison_line["預測"],
        mode="lines",
        name="預測值",
        line=dict(color=REF_COLOR, width=2, dash="dash"),
    )
)
fig.update_layout(yaxis_title=label("Q_COOL"), hovermode="x unified")
date_axis(fig)
show(fig, "冷凍噸：實際值 vs 預測值", f"in-sample：{metric_text}", legend="bottom")

In [8]:
# 點越貼近虛線（y = x）預測越準
axis_low = comparison.min().min()
axis_high = comparison.max().max()

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=comparison["實際"],
        y=comparison["預測"],
        mode="markers",
        name="每小時",
        marker=dict(
            size=8,
            color=COLORS["Q_COOL"],
            opacity=0.8,
            line=dict(width=1, color="white"),
        ),
        text=comparison.index.strftime("%m/%d %H:%M"),
        hovertemplate=(
            "%{text}<br>實際: %{x:.1f} RT<br>預測: %{y:.1f} RT<extra></extra>"
        ),
    )
)
fig.add_trace(
    go.Scatter(
        x=[axis_low, axis_high],
        y=[axis_low, axis_high],
        mode="lines",
        name="理想預測線 y = x",
        line=dict(color=REF_COLOR, width=2, dash="dash"),
    )
)
fig.update_layout(
    xaxis_title=f"實際 {label('Q_COOL')}",
    yaxis_title=f"預測 {label('Q_COOL')}",
)
show(
    fig,
    "冷凍噸：實際值 vs 預測值（散佈圖）",
    f"in-sample：{metric_text}",
    legend="bottom",
)

### 觀察重點

- **R² = 0.99**：訓練與召回用同一批資料，不代表沒看過的日子也這麼準
- **高度依賴前一天負荷**：天氣變化造成的負荷差異，模型看不到